# صوت فيديو نهايات الدالة الأسية بـ ElevenLabs
شغّل الخلايا بالترتيب.
1. **الخلية 2:** ضع مفتاح API في أسرار Colab (أيقونة المفتاح 🔑 على اليسار) باسم `ELEVEN_API_KEY`. **لا ترسل المفتاح لأحد.**
2. **الخلية 4:** تعرض الأصوات المتاحة في حسابك. انسخ `voice_id` للصوت الذي يعجبك في `VOICE_ID`.
3. **الخلية 4:** تعرض أيضًا النماذج، ويُختار **Eleven v4** تلقائيًا.
4. **الخلية 5:** تجربة جملة واحدة.
5. **الخلية 6:** الصوت كاملًا. كل جملة تُحفظ وحدها، فإن أعدت التشغيل لا تُولَّد إلا الجمل المذكورة في `REDO` أو التي لم تُولَّد بعد.
6. أرسل لي `explim_voice.zip` كما هو.


In [ ]:
!pip -q install requests pydub "audioop-lts; python_version>='3.13'"


In [ ]:
from google.colab import userdata
API_KEY = userdata.get("ELEVEN_API_KEY")
MODEL = "eleven_v4"             # فارغ = يختار Eleven v4 تلقائيًا (الخلية 4 تعرض النماذج). أو ضع مثلًا "eleven_multilingual_v2"
VOICE_ID = "G1HOkzin3NMwRHSq60UI"          # ضع هنا voice_id من الخلية 4
SETTINGS = {"stability": 0.5, "similarity_boost": 0.8, "style": 0.3, "use_speaker_boost": True, "speed": 0.95}
GAP = 0.6              # صمت بين الجمل (ثوانٍ)
LEAD = 0.4
REDO = list(range(1, 42))              # أرقام الجمل التي تريد إعادة توليدها (تبدأ من 1)، مثلًا [3, 7]
LINES = [
 "أهم نهايات الدالة الأسية في مواضيع البكالوريا، من ألفين وعشرين إلى ألفين وستة وعشرين، وكلها تُحسب بخمس طرائق فقط.",
 "نحلّها معًا خطوة بخطوة، مع إزالة كل حالات عدم التعيين.",
 "نبدأ بالتذكير: نهاية أُو أُسْ إِكْسْ عند زائد ما لا نهاية هي زائد ما لا نهاية، وعند ناقص ما لا نهاية هي صفر.",
 "والتزايد المقارن: أُو أُسْ إِكْسْ على إِكْسْ يؤول إلى زائد ما لا نهاية، وإِكْسْ في أُو أُسْ إِكْسْ يؤول إلى صفر عند ناقص ما لا نهاية.",
 "وحالات عدم التعيين أربع: ما لا نهاية ناقص ما لا نهاية، صفر في ما لا نهاية، ما لا نهاية على ما لا نهاية، وصفر على صفر.",
 "الطريقة الأولى: الحساب المباشر، عندما لا توجد حالة عدم تعيين.",
 "بكالوريا ألفين وثلاثة وعشرين، علوم تجريبية: الدالة أَفْ معرّفة بالجداء: إِكْسْ ناقص واحد، في أُو أُسْ اثنين إِكْسْ ناقص واحد.",
 "عند زائد ما لا نهاية: العاملان يؤولان إلى زائد ما لا نهاية، إذن الجداء يؤول إلى زائد ما لا نهاية.",
 "وعند ناقص ما لا نهاية: إِكْسْ ناقص واحد يؤول إلى ناقص ما لا نهاية، والعامل الثاني يؤول إلى ناقص واحد، إذن النهاية زائد ما لا نهاية.",
 "بكالوريا ألفين وعشرين، علوم تجريبية: أَفْ من إِكْسْ يساوي ناقص واحد زائد، اثنين أُو أُسْ إِكْسْ، على أُو أُسْ إِكْسْ ناقص إِكْسْ.",
 "عند ناقص ما لا نهاية: البسط يؤول إلى صفر، والمقام إلى زائد ما لا نهاية، فالكسر يؤول إلى صفر، والنهاية ناقص واحد.",
 "الطريقة الثانية: عدد غير معدوم على صفر، والحاسم هو إشارة المقام.",
 "بكالوريا ألفين وستة وعشرين، علوم تجريبية: أَفْ من إِكْسْ يساوي واحد زائد، خمسة ناقص أربعة أُو أُسْ إِكْسْ، على أُو أُسْ اثنين إِكْسْ ناقص واحد.",
 "عند الصفر: البسط يؤول إلى واحد، والمقام يؤول إلى صفر.",
 "المقام موجب إذا كان إِكْسْ موجبًا، وسالب إذا كان إِكْسْ سالبًا، إذن النهاية زائد ما لا نهاية على اليمين، وناقص ما لا نهاية على اليسار.",
 "ومنه المستقيم ذو المعادلة إِكْسْ يساوي صفر مقارب عمودي.",
 "الطريقة الثالثة: ما لا نهاية على ما لا نهاية، نقسم البسط والمقام على الحد الأسي الأقوى.",
 "نفس دالة ألفين وستة وعشرين عند زائد ما لا نهاية: نقسم البسط والمقام على أُو أُسْ اثنين إِكْسْ.",
 "البسط يؤول إلى صفر والمقام إلى واحد، فالكسر يؤول إلى صفر، إذن النهاية واحد.",
 "بكالوريا ألفين وعشرين عند زائد ما لا نهاية: نخرج أُو أُسْ إِكْسْ عاملًا مشتركًا في المقام.",
 "يصبح الكسر: اثنين على واحد ناقص إِكْسْ أُو أُسْ ناقص إِكْسْ، وبالتزايد المقارن النهاية واحد.",
 "بكالوريا ألفين وستة وعشرين، رياضيات: نقسم على أُو أُسْ إِكْسْ فنحصل على إِكْسْ مربع على واحد زائد أُو أُسْ ناقص إِكْسْ، والنهاية زائد ما لا نهاية.",
 "الطريقة الرابعة: ما لا نهاية ناقص ما لا نهاية، نخرج الحد الأقوى عاملًا مشتركًا.",
 "بكالوريا ألفين وخمسة وعشرين، علوم تجريبية: أُو أُسْ اثنين إِكْسْ ناقص أُو أُسْ إِكْسْ ناقص إِكْسْ ناقص اثنين، عند زائد ما لا نهاية.",
 "نخرج أُو أُسْ اثنين إِكْسْ عاملًا: ما داخل القوس يؤول إلى واحد، إذن النهاية زائد ما لا نهاية.",
 "بكالوريا ألفين وأربعة وعشرين، علوم تجريبية: ناقص اثنين إِكْسْ زائد ثلاثة، ناقص إِكْسْ أُو أُسْ ناقص إِكْسْ زائد واحد، عند ناقص ما لا نهاية.",
 "انتبه للخطأ الشائع: زائد ما لا نهاية ناقص ما لا نهاية لا يساوي صفرًا.",
 "نخرج إِكْسْ عاملًا: القوس يؤول إلى ناقص ما لا نهاية، وإِكْسْ إلى ناقص ما لا نهاية، إذن النهاية زائد ما لا نهاية.",
 "الطريقة الخامسة: صفر في ما لا نهاية، ونستعمل التزايد المقارن.",
 "نفس دالة ألفين وأربعة وعشرين عند زائد ما لا نهاية: إِكْسْ أُو أُسْ ناقص إِكْسْ يؤول إلى صفر، إذن النهاية ناقص ما لا نهاية.",
 "بكالوريا ألفين وستة وعشرين، تقني رياضي: واحد زائد، واحد ناقص إِكْسْ ناقص إِكْسْ مربع، في أُو أُسْ ناقص إِكْسْ، عند زائد ما لا نهاية.",
 "ننشر الجداء، وبالتزايد المقارن كل الحدود تؤول إلى صفر، إذن النهاية واحد.",
 "بكالوريا ألفين وستة وعشرين، رياضيات، عند ناقص ما لا نهاية: إِكْسْ مربع أُو أُسْ إِكْسْ يؤول إلى صفر، إذن النهاية صفر.",
 "تطبيق مهم: المستقيم المقارب المائل، نبيّن أن نهاية الفرق بين أَفْ والمستقيم تساوي صفرًا.",
 "بكالوريا ألفين وثلاثة وعشرين: الفرق يساوي إِكْسْ ناقص واحد في أُو أُسْ اثنين إِكْسْ، ويؤول إلى صفر عند ناقص ما لا نهاية، إذن المستقيم ذو المعادلة ي يساوي ناقص إِكْسْ زائد واحد مقارب مائل.",
 "بكالوريا ألفين وواحد وعشرين: الفرق يساوي ناقص إِكْسْ زائد واحد في أُو أُسْ ناقص إِكْسْ ناقص واحد، ويؤول إلى صفر عند زائد ما لا نهاية، إذن المستقيم ي يساوي إِكْسْ مقارب مائل.",
 "والآن حاول بنفسك: بكالوريا ألفين واثنين وعشرين، تقني رياضي، احسب هذه النهاية، وأوقف الفيديو.",
 "الجواب: زائد ما لا نهاية، والحل المفصل موجود في الملف.",
 "الخلاصة: إن لم توجد حالة عدم تعيين نحسب مباشرة، وعدد على صفر ننظر إلى إشارة المقام، وما لا نهاية على ما لا نهاية نقسم على الحد الأقوى.",
 "وما لا نهاية ناقص ما لا نهاية نخرج الحد الأقوى عاملًا، وصفر في ما لا نهاية نستعمل التزايد المقارن.",
 "كل تمارين البكالوريا في الدالة الأسية، من ألفين وثمانية إلى ألفين وستة وعشرين، بالحل المفصل، موجودة في ملف تمارين البكالوريا. راسلنا على واتساب أو على صفحتنا."
]
# تصحيح النطق: الكلمة كما تظهر على الشاشة -> الكتابة التي تُنطق (تشكيل أو كتابة صوتية).
# كلمة واحدة مقابل كلمة واحدة حتى يبقى توقيت الكتابة صحيحًا.
FIX = {
 "الثانية": "الثَّانِيَةُ",
 "الأولى": "الأُولَى",
 "العلمية": "العِلْمِيَّة",
 "السنة": "السَّنَةُ",
 "الجداء": "الجُدَاء",
 "السلمي": "السُّولَّمِي",
 "الموجهة": "المُوَجَّهَة",
 "الشعاعي": "الشُّعَاعِي",
 "المرجعية": "المَرْجِعِيَّة",
 "عموميات": "عُمُومِيَّات",
 "التكاملي": "التَّكَامُلِي",
 "المقارن": "المُقَارَن",
 "المستوية": "المُسْتَوِيَة",
 "المثلثات": "المُثَلَّثَات",
 "التزايد": "التَّزَايُد",
 "الاستمرارية": "الاِسْتِمْرَارِيَّة",
 "أستاذ": "أُسْتَاذِ",
 "تعليم": "تَعْلِيمٍ",
 "يدرّس": "يُدَرِّسُ",
 "يدرس": "يَدْرُسُ",
 "إليك": "إِلَيْكَ",
 "الشعب": "الشُّعَبُ",
 "ولشعبة": "وَلِشُعْبَة",
 "شعبة": "شُعْبَة",
 "الأسية": "الأُسِّيَّة",
 "صيغ": "صِيَغٍ",
 "ملفًا": "مِلَفًّا",
 "ملفات": "مِلَفَّات",
 "الملفات": "المِلَفَّات",
 "والملف": "وَالمِلَفُّ",
 "ولاتاك": "وLaTeX",
 "ووورد": "وَوُورْدْ",
 "ملخصات": "مُلَخَّصَات",
 "ومذكرات": "وَمُذَكِّرَات",
 "مذكرة": "مُذَكِّرَة",
 "وملخص": "وَمُلَخَّص",
 "ملخص": "مُلَخَّص",
 "المرجح": "المُرَجِّح",
 "المركبة": "المُرَكَّبَة",
 "المتتاليات": "المُتَتَالِيَات",
 "اللوغاريتمية": "اللُّوغَارِيتْمِيَّة",
 "الاشتقاقية": "الاِشْتِقَاقِيَّة",
 "والاستمرارية": "وَالاِسْتِمْرَارِيَّة",
 "جذع": "جِذْع",
 "اقتناء": "اقْتِنَاء",
 "فايسبوك": "فَايْسْبُوكْ",
 "إنستغرام": "إِنْسْتَغْرَامْ",
 "واتساب": "وَاتْسَابْ",
 "ثانوي": "ثَانَوِي",
}
# تصحيح خاص بجملة واحدة (رقمها يبدأ من 1): يغلب FIX في تلك الجملة فقط
LINE_FIX = {}
def spoken(line, i=0):
    fx = {**FIX, **LINE_FIX.get(i, {})}
    out = []
    for w in line.split():
        core = w.strip("،:؟.!")
        out.append(fx[core] + w[len(w.rstrip("،:؟.!")):] if core in fx else w)
    return " ".join(out)

# جمل تُولَّد على جزأين (الجزء الأول قصير وحده يُنطق صحيحًا): رقم الجملة -> عدد كلمات الجزء الأول
SPLIT_AT = {}   # «السَّنَةُ الثَّانِيَةُ» و«السَّنَةُ الأُولَى» وحدهما كما في التجربة رقم 1


In [ ]:
import requests
print("المفتاح موجود:", bool(API_KEY))
r = requests.get("https://api.elevenlabs.io/v1/voices", headers={"xi-api-key": API_KEY})
print("الحالة:", r.status_code)
j = r.json()
if "voices" not in j:
    print(j)
for v in j.get("voices", []):
    print(v["voice_id"], "|", v["name"], "|", v.get("category"), "|", v.get("labels", {}))

# النماذج المتاحة لحسابك: نختار Eleven v4 إن لم تحدد MODEL
m = requests.get("https://api.elevenlabs.io/v1/models", headers={"xi-api-key": API_KEY}).json()
print("\nالنماذج:")
for x in m if isinstance(m, list) else []:
    print(" ", x.get("model_id"), "|", x.get("name"), "| tts:", x.get("can_do_text_to_speech"))
if not MODEL:
    v4 = [x["model_id"] for x in (m if isinstance(m, list) else []) if x.get("can_do_text_to_speech") and ("v4" in x.get("model_id", "").lower() or "v4" in x.get("name", "").lower())]
    MODEL = v4[0] if v4 else "eleven_multilingual_v2"
print("\nالنموذج المستعمل:", MODEL)


In [ ]:
import base64, io, os, json
from pydub import AudioSegment
from IPython.display import Audio, display
os.makedirs("parts", exist_ok=True)

def _post(url, text, settings):
    return requests.post(url, headers={"xi-api-key": API_KEY}, json={"text": text, "model_id": MODEL, "voice_settings": settings, "seed": 7})

def tts(text):
    base = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}"
    simple = {k: SETTINGS[k] for k in ("stability", "similarity_boost") if k in SETTINGS}
    r = None
    for url in (base + "/with-timestamps?output_format=mp3_44100_128", base + "?output_format=mp3_44100_128"):
        for st in (SETTINGS, simple, {"stability": 0.5}):
            r = _post(url, text, st)
            if r.ok: break
        if r.ok: break
    if not r.ok: print(r.status_code, r.text[:500]); r.raise_for_status()
    if r.headers.get("content-type", "").startswith("application/json"):
        j = r.json(); seg = AudioSegment.from_file(io.BytesIO(base64.b64decode(j["audio_base64"])), format="mp3")
        a = j.get("alignment") or j.get("normalized_alignment")
    else:                                   # بدون توقيت: نقدّر توقيت الكلمات من طولها
        seg = AudioSegment.from_file(io.BytesIO(r.content), format="mp3"); a = None
    words, cur = [], None
    if a:
        for ch, s, e in zip(a["characters"], a["character_start_times_seconds"], a["character_end_times_seconds"]):
            if ch.isspace():
                if cur: words.append(cur); cur = None
            elif cur is None: cur = {"text": ch, "start": s, "end": e}
            else: cur["text"] += ch; cur["end"] = e
        if cur: words.append(cur)
    else:
        ws = text.split(); tot = sum(len(x) for x in ws); t = 0.15; dur = len(seg) / 1000 - 0.3
        for x in ws:
            dt = dur * len(x) / tot; words.append({"text": x, "start": t, "end": t + dt}); t += dt
    return seg, words

seg, w = tts(spoken(LINES[2], 3))   # تجربة الجملة 7 (السنة الثانية ثانوي، الشعب العلمية)
print(len(w), "كلمة"); display(Audio(seg.export(format="mp3").read()))


In [ ]:
# اختبار طرق كتابة كلمة: استمع لكل صيغة، واختر الصحيحة ثم ضعها في FIX
VARIANTS = ["أُهْ أُسْ إيكْسْ", "أَهْ أُسْ إيكْسْ", "ئُو أُسُّو إيكْس", "أُو، أُسْ، إيكْس", "eu os iks", "e puissance x", "أُهْ", "أُسْ", "إيكْسْ", "أُوسْ"]
for k, v in enumerate(VARIANTS, 1):
    s_, _ = tts(v); print(k, v); display(Audio(s_.export(format="mp3").read()))


In [ ]:
import zipfile
from google.colab import files
full = AudioSegment.silent(duration=int(LEAD * 1000)); out = []
for i, line in enumerate(LINES, 1):
    mp3, js = f"parts/{i:02d}.mp3", f"parts/{i:02d}.json"
    if i in REDO or not os.path.exists(mp3):
        parts_ = [line] if i not in SPLIT_AT else [" ".join(line.split()[:SPLIT_AT[i]]), " ".join(line.split()[SPLIT_AT[i]:])]
        seg, words = AudioSegment.empty(), []
        for k_, p_ in enumerate(parts_):
            print(i, "يُرسَل:", spoken(p_, i))
            s_, w_ = tts(spoken(p_, i)); off = len(seg) / 1000
            words += [{**x, "start": x["start"] + off, "end": x["end"] + off} for x in w_]
            seg += s_ + (AudioSegment.silent(duration=60) if k_ + 1 < len(parts_) else AudioSegment.empty())
        seg.export(mp3, format="mp3", bitrate="192k")
        json.dump(words, open(js, "w"), ensure_ascii=False)
        print(i, "توليد", round(len(seg) / 1000, 1), "ث")
    seg = AudioSegment.from_file(mp3); words = json.load(open(js))
    shown = line.split()
    if len(shown) == len(words):            # نعرض الكلمات كما في السيناريو (قبل تصحيح النطق)
        for w_, s_ in zip(words, shown): w_["text"] = s_
    t0 = len(full) / 1000
    out.append({"i": i, "text": line, "start": t0, "end": t0 + len(seg) / 1000,
                "words": [{**w_, "start": w_["start"] + t0, "end": w_["end"] + t0} for w_ in words]})
    full += seg + AudioSegment.silent(duration=int(GAP * 1000))
full.export("voice.mp3", format="mp3", bitrate="192k")
json.dump({"voice": VOICE_ID, "model": MODEL, "duration": len(full) / 1000, "lines": out}, open("timings.json", "w"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("explim_voice.zip", "w") as z:
    z.write("voice.mp3"); z.write("timings.json")
print("المدة:", round(len(full) / 1000), "ثانية"); display(Audio("voice.mp3"))
files.download("explim_voice.zip")
